# Baseline: QLoRA Text-to-SVG Generation

**NYU Deep Learning — Spring 2026 Midterm Kaggle Competition**

This notebook implements a complete baseline pipeline in **two phases**:

**Phase A — Training** (requires internet & GPU):

1. Environment Setup — dependencies, seeds, configuration
2. SVG Utilities — competition-compliant validation, post-processing, fallback
3. Data Pipeline — multi-source loading, normalization, quality filtering
4. Model Setup — Qwen 2B + QLoRA via Unsloth
5. Training — SFT with chat-formatted (prompt, SVG) pairs

**Phase B — Inference & Submission** (can run offline on Kaggle):

6. Inference & Submission — generate, validate, export `submission.csv`

> **For Kaggle Code Submission:** split at the phase boundary. Upload the trained adapter as a Kaggle dataset, then create a separate offline notebook that only runs Phase B.

In [ ]:
%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml cairosvg

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ET.register_namespace('', 'http://www.w3.org/2000/svg')
ET.register_namespace('xlink', 'http://www.w3.org/1999/xlink')

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
CONFIG = {
    # Model
    'model_name': 'unsloth/Qwen2.5-3B-Instruct',
    'max_seq_length': 2048,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 16,
    'lora_dropout': 0,
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],

    # Training
    'learning_rate': 2e-4,
    'num_train_epochs': 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 32,
    'warmup_ratio': 0.05,
    'weight_decay': 0.01,
    'logging_steps': 20,
    'eval_steps': 100,
    'save_steps': 200,

    # Data
    'max_train_samples_per_source': 12000,
    'max_svg_train_chars': 2500,
    'eval_size': 0.02,

    # SVG constraints (competition rules)
    'max_svg_length': 16000,
    'max_path_count': 256,
    'canvas_size': 256,

    # Inference
    'inference_batch_size': 4,
    'max_new_tokens': 768,
    'temperature': 0.3,
    'top_p': 0.8,
    'repetition_penalty': 1.05,

    
    # Paths — Colab + Google Drive
    #'train_data_path': '/content/drive/MyDrive/midterm/train.csv',
    #'output_dir': '/content/drive/MyDrive/midterm/qwen25_3b_svg_lora',
    #'test_prompts_path': '/content/drive/MyDrive/midterm/test.csv',
    #'submission_path': '/content/drive/MyDrive/midterm/submission.csv',
    
    
    # Paths — Kaggle
    'train_data_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv',
    'output_dir': '/kaggle/working/qwen25_3b_svg_lora',
    'test_prompts_path': '/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv',
    'submission_path': '/kaggle/working/submission.csv',
}

CONFIG

---
## 2. SVG Utilities

Competition-compliant validation, post-processing, and fallback.

Key constraints: 256×256 canvas, ≤16 000 chars, ≤256 paths, whitelist-only tags, no scripts/events/animation.

In [ ]:
ALLOWED_TAGS = frozenset({
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse', 'line', 'polyline',
    'polygon', 'defs', 'use', 'symbol', 'clipPath', 'mask',
    'linearGradient', 'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
})


def _strip_ns(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def _count_paths(root):
    return sum(1 for e in root.iter() if _strip_ns(e.tag) == 'path')


def _collect_bad_tags(root):
    return {_strip_ns(e.tag) for e in root.iter() if _strip_ns(e.tag) not in ALLOWED_TAGS}


def _has_event_handlers(root):
    for e in root.iter():
        for attr in e.attrib:
            local = attr.split('}')[-1] if '}' in attr else attr
            if local.lower().startswith('on'):
                return True
    return False


def validate_svg(svg_text, max_length=16000, max_paths=256):
    """
    Competition-compliant SVG validation.
    Returns (is_valid, reason).
    """
    if not svg_text or not isinstance(svg_text, str):
        return False, 'empty'
    if len(svg_text) > max_length:
        return False, f'too_long ({len(svg_text)})'
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return False, f'parse_error: {e}'
    if _strip_ns(root.tag) != 'svg':
        return False, f'root_not_svg ({root.tag})'
    bad = _collect_bad_tags(root)
    if bad:
        return False, f'bad_tags: {bad}'
    if _has_event_handlers(root):
        return False, 'event_handlers'
    n = _count_paths(root)
    if n > max_paths:
        return False, f'too_many_paths ({n})'
    return True, 'ok'


def sanitize_svg(svg_text):
    """
    Post-process SVG: fix root attributes, strip disallowed elements/attributes.
    Returns cleaned SVG string or empty string on failure.
    """
    if not svg_text:
        return ''
    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ''
    if _strip_ns(root.tag) != 'svg':
        return ''
    if '{' not in root.tag:
        root.tag = '{http://www.w3.org/2000/svg}svg'
    root.attrib.pop('xmlns', None)
    root.set('width', '256')
    root.set('height', '256')
    root.set('viewBox', '0 0 256 256')
    _remove_bad_elements(root)
    _remove_event_attrs(root)
    return ET.tostring(root, encoding='unicode')


def _remove_bad_elements(elem):
    to_remove = [c for c in elem if _strip_ns(c.tag) not in ALLOWED_TAGS]
    for c in to_remove:
        elem.remove(c)
    for c in elem:
        _remove_bad_elements(c)


def _remove_event_attrs(root):
    for e in root.iter():
        bad = [a for a in e.attrib
               if (a.split('}')[-1] if '}' in a else a).lower().startswith('on')]
        for a in bad:
            del e.attrib[a]


def fallback_svg(prompt=''):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" '
        'viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="gray"/>'
        '</svg>'
    )


_ok, _reason = validate_svg(fallback_svg())
print(f'Fallback SVG valid: {_ok} ({_reason}), length: {len(fallback_svg())} chars')

---
## 3. Data Pipeline

Load competition `train.csv`, normalize SVGs to 256×256, and apply quality filters.

In [ ]:
_PROMPT_PREFIX_RE = re.compile(
    r'^(?:Generate|Create|Write|Make|Draw|Produce)\s+'
    r'(?:(?:the\s+|an?\s+)?(?:SVG|svg)\s+(?:code|image|graphic)\s+)?'
    r'(?:for\s+)?(?:an?\s+image\s+(?:that|which)\s+looks?\s+like[:\s]*)?',
    re.IGNORECASE,
)
_PROMPT_SUFFIX_RE = re.compile(
    r"[\.\s]*(?:Don'?t|do\s+not)\s+use\s+markdown.*$",
    re.IGNORECASE,
)


def _clean_prompt(prompt):
    """Strip instruction wrappers, keeping only the visual description."""
    prompt = _PROMPT_PREFIX_RE.sub('', prompt)
    prompt = _PROMPT_SUFFIX_RE.sub('', prompt)
    return prompt.strip()


def _quality_filter(example):
    """Keep only samples with valid, reasonably-sized SVGs."""
    svg = example.get('svg', '')
    prompt = example.get('prompt', '')
    if not svg or not prompt:
        return False
    if len(svg) > CONFIG['max_svg_train_chars']:
        return False
    ok, _ = validate_svg(svg)
    return ok

In [ ]:
# ── Skip-training guard ──
import os as _os
_KAGGLE_INPUT_LORA = '/kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1'
_adapter_working = _os.path.join(CONFIG["output_dir"], "adapter_config.json")
_adapter_kaggle  = _os.path.join(_KAGGLE_INPUT_LORA,   "adapter_config.json")
if _os.path.exists(_adapter_kaggle):
    SKIP_TRAINING = True
    print(f"[SKIP] Kaggle input adapter found: {_KAGGLE_INPUT_LORA}")
    print("       Cells 11, 13, 14 will be skipped. Jump to Cell 17 for inference.")
elif _os.path.exists(_adapter_working):
    SKIP_TRAINING = True
    print(f"[SKIP] Working-dir adapter found: {CONFIG['output_dir']}")
    print("       Cells 11, 13, 14 will be skipped. Jump to Cell 17 for inference.")
else:
    SKIP_TRAINING = False
    print("[TRAIN] No adapter found — will train from scratch.")

# ── Load competition train.csv as primary data source ──
comp_df = pd.read_csv(CONFIG['train_data_path'])
print(f'Competition train.csv: {len(comp_df)} rows, columns={list(comp_df.columns)}')

comp_ds = Dataset.from_pandas(comp_df[['prompt', 'svg']])


def _normalize_comp(example):
    prompt = str(example.get('prompt', '')).strip()
    prompt = _clean_prompt(prompt)
    svg = sanitize_svg(str(example.get('svg', '')))
    return {'prompt': prompt, 'svg': svg}


comp_ds = comp_ds.map(_normalize_comp, remove_columns=comp_ds.column_names,
                      desc='normalizing train.csv')
before = len(comp_ds)
comp_ds = comp_ds.filter(_quality_filter, desc='filtering train.csv')
print(f'  Usable: {len(comp_ds)} rows (dropped {before - len(comp_ds)})')

max_samples = CONFIG['max_train_samples_per_source']
if max_samples and len(comp_ds) > max_samples:
    comp_ds = comp_ds.shuffle(seed=SEED).select(range(max_samples))
    print(f'  Subsampled to {len(comp_ds)} rows')

train_raw = comp_ds.shuffle(seed=SEED)
splits = train_raw.train_test_split(test_size=CONFIG['eval_size'], seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

print(f'\nTrain: {len(train_ds)} | Eval: {len(eval_ds)}')
if len(train_ds) > 0:
    print(f'Sample prompt: {train_ds[0]["prompt"][:120]}')
    print(f'Sample SVG (first 200): {train_ds[0]["svg"][:200]}')
else:
    raise RuntimeError(
        'Training set is empty after filtering. '
        'Check max_svg_train_chars / validate_svg settings.'
    )

In [ ]:
SYSTEM_PROMPT = (
    'You generate compact, valid SVG code from text descriptions. '
    'Output only a single <svg> element with xmlns="http://www.w3.org/2000/svg", '
    'width="256", height="256", viewBox="0 0 256 256". '
    'No explanation, no markdown — only SVG code.'
)


def format_sft(example):
    text = (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{example["prompt"]}<|im_end|>\n'
        '<|im_start|>assistant\n'
        f'{example["svg"]}<|im_end|>'
    )
    return {'text': text}


train_text = train_ds.map(format_sft, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft, remove_columns=eval_ds.column_names)

print('Formatted sample (first 500 chars):')
print(train_text[0]['text'][:500])

---
## 4. Model Setup — Qwen 2B + QLoRA (Unsloth)

In [ ]:
if not SKIP_TRAINING:
    from unsloth import FastLanguageModel
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CONFIG['model_name'],
        max_seq_length=CONFIG['max_seq_length'],
        dtype=None,
        load_in_4bit=True,
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=CONFIG['lora_r'],
        lora_alpha=CONFIG['lora_alpha'],
        lora_dropout=CONFIG['lora_dropout'],
        bias='none',
        target_modules=CONFIG['target_modules'],
        use_gradient_checkpointing='unsloth',
        random_state=SEED,
    )
    
    model.print_trainable_parameters()

else:
    print("[SKIP] Model setup skipped (adapter already exists).")


---
## 5. Training

In [ ]:
if not SKIP_TRAINING:
    from transformers import TrainingArguments, Trainer as _HFTrainer
    from trl import SFTTrainer
    
    # ============ Fix: Unsloth / transformers >=4.46 compatibility ============
    # transformers passes num_items_in_batch as int; Unsloth calls .mean() on it.
    # Strategy A — class-level patch on Trainer.training_step (survives compiled cache)
    _orig_trainer_step = _HFTrainer.training_step
    
    def _safe_training_step(self, model, inputs, num_items_in_batch=None):
        if isinstance(num_items_in_batch, (int, float)):
            num_items_in_batch = torch.tensor(float(num_items_in_batch))
        return _orig_trainer_step(self, model, inputs, num_items_in_batch)
    
    _HFTrainer.training_step = _safe_training_step
    print("[patch] Trainer.training_step wrapped for int→tensor fix")
    # ==========================================================================
    
    training_args = TrainingArguments(
        output_dir=CONFIG['output_dir'],
        num_train_epochs=CONFIG['num_train_epochs'],
        per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
        gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
        learning_rate=CONFIG['learning_rate'],
        warmup_ratio=CONFIG['warmup_ratio'],
        weight_decay=CONFIG['weight_decay'],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=CONFIG['logging_steps'],
        eval_strategy='steps',
        eval_steps=CONFIG['eval_steps'],
        save_steps=CONFIG['save_steps'],
        save_total_limit=2,
        report_to='none',
        optim='paged_adamw_8bit',
        lr_scheduler_type='cosine',
        seed=SEED,
    )
    
    _resp_tpl = '<|im_start|>assistant\n'
    print(f'[info] response_template = {_resp_tpl!r}')
    
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_text,
        eval_dataset=eval_text,
        dataset_text_field='text',
        max_seq_length=CONFIG['max_seq_length'],
        packing=False,
        args=training_args,
        response_template=_resp_tpl,
    )
    
    # Strategy B — disable num_items_in_batch from being passed at all
    trainer.model_accepts_loss_kwargs = False
    print("[patch] model_accepts_loss_kwargs = False  →  num_items_in_batch disabled")
    
    train_result = trainer.train()
    print(train_result)

else:
    print("[SKIP] Training skipped (adapter already exists).")


In [ ]:
if not SKIP_TRAINING:
    os.makedirs(CONFIG['output_dir'], exist_ok=True)
    trainer.save_model(CONFIG['output_dir'])
    tokenizer.save_pretrained(CONFIG['output_dir'])
    print(f'Adapter + tokenizer saved to {CONFIG["output_dir"]}')

else:
    print("[SKIP] Save skipped (adapter already exists).")


In [ ]:
if not SKIP_TRAINING:
    import gc
    
    # Free training artifacts — optimizer states, gradients, cached activations
    del trainer, training_args
    gc.collect()
    torch.cuda.empty_cache()
    
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"GPU memory after cleanup: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

else:
    import gc; gc.collect()
    print("[SKIP] Cleanup skipped (no training artifacts to free).")


---
## Phase B: Inference & Submission

> Everything below this line can run **offline** on Kaggle.
> For the actual Kaggle code submission, create a separate notebook that loads the pre-trained adapter from a Kaggle dataset and only executes the cells below.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Cell 17 综合修复方案 v3
# 用于替换 baseline-qwen2-5.ipynb 的 Cell 17 + Smoke Test
#
# 解决的三个问题：
#
# P1: RuntimeError — Unsloth fast inference 在 Qwen2.5 上 cos/sin
#     缓存维度不对齐 (analysis_to_this_error.md 根因分析)
#     → 修复：不用 Unsloth fast path，走干净 HF generate
#
# P2: valid=0 fallback=1000 — 所有生成都走了 fallback
#     → 修复：添加详细诊断输出，定位是 extract/sanitize/validate
#       哪一步失败
#
# P3: 24.5s/prompt 6.8h 总耗时 — 从不提前停止
#     → 修复：正确清理 generation_config (消除 max_length=32768
#       冲突)，确保 EOS 生效，SDPA 加速
#
# 用法：复制到 Colab / Kaggle 的 Cell 17 位置
# ═══════════════════════════════════════════════════════════════════

import gc, importlib, time, re
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
from peft import PeftModel

# ── 路径配置：自动选择 LoRA adapter 路径 ──
_KAGGLE_INPUT_LORA = '/kaggle/input/models/sxunyu/qwen25-3b-svg-lora/transformers/default/1'
if os.path.exists(os.path.join(_KAGGLE_INPUT_LORA, "adapter_config.json")):
    _LORA_DIR = _KAGGLE_INPUT_LORA
    print(f"[PATH] Using Kaggle input LoRA: {_LORA_DIR}")
else:
    _LORA_DIR = CONFIG['output_dir']
    print(f"[PATH] Using working-dir LoRA: {_LORA_DIR}")

# ════════════════════════════════════════════════════════════
# Phase 1: 环境清理 — 无条件释放训练态对象
# ════════════════════════════════════════════════════════════

# 1a. 清除 Unsloth 对 qwen2 模块的全局补丁
import transformers.models.qwen2.modeling_qwen2 as _qwen2
importlib.reload(_qwen2)
print('[P1] Reloaded qwen2 module — Unsloth forward patches removed')

# 1b. 删除所有训练态模型对象（不复用！这是分析文档的核心建议）
try:
    del model
except NameError:
    pass
try:
    del base_model
except NameError:
    pass
try:
    del trainer
except NameError:
    pass
try:
    del peft_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
_free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'[P1] GPU cleared. Free VRAM: {_free_gb:.1f} GB')

# ════════════════════════════════════════════════════════════
# Phase 2: 加载干净模型 + LoRA 合并
# ════════════════════════════════════════════════════════════

# 2a. 从 HuggingFace 加载基础模型，显式指定 attn_implementation
#     这比 post-hoc 设 config._attn_implementation 更可靠
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG['model_name'],
    device_map='auto',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print(f'[P2] Base model loaded: {type(base_model).__name__}')

# 2b. 全量替换 __class__ → 干净的 HF 类（belt-and-suspenders）
#     即使 importlib.reload 已还原模块级定义，from_pretrained 可能
#     通过缓存的 AUTO_MODEL_MAPPING 引用了旧的（被 Unsloth 补丁过的）类
base_model.__class__ = _qwen2.Qwen2ForCausalLM
base_model.model.__class__ = _qwen2.Qwen2Model
_n_layers = len(base_model.model.layers)
for _layer in base_model.model.layers:
    _layer.__class__ = _qwen2.Qwen2DecoderLayer
    _layer.self_attn.__class__ = _qwen2.Qwen2Attention
    _layer.mlp.__class__ = _qwen2.Qwen2MLP
    _layer.input_layernorm.__class__ = _qwen2.Qwen2RMSNorm
    _layer.post_attention_layernorm.__class__ = _qwen2.Qwen2RMSNorm
base_model.model.norm.__class__ = _qwen2.Qwen2RMSNorm
base_model.config._attn_implementation = 'sdpa'
print(f'[P2] Class-swapped {_n_layers} layers → clean HF + sdpa')

# 2c. 加载 LoRA adapter 并合并进基础权重
#     merge_and_unload() 消除 PeftModel 包装的 252 层 per-token
#     Python 调度开销（之前 9.5 tok/s 慢速的主因之一）
_peft = PeftModel.from_pretrained(base_model, _LORA_DIR)
model = _peft.merge_and_unload()
del _peft, base_model
gc.collect()
# 新版 peft 在 merge_and_unload() 后仍可能在 model 上残留 peft_config 属性，
# 该属性对推理无影响，但会导致 hasattr 检查误报。显式删除。
if hasattr(model, 'peft_config'):
    del model.peft_config
print(f'[P2] LoRA merged into base weights — plain {type(model).__name__}')

# 2d. 清除所有残留的 Unsloth forward hooks
_hooks_cleared = 0
for _m in model.modules():
    _hooks_cleared += len(_m._forward_hooks) + len(_m._forward_pre_hooks)
    _m._forward_hooks.clear()
    _m._forward_pre_hooks.clear()
print(f'[P2] Cleared {_hooks_cleared} residual hooks')

model.eval()

# ════════════════════════════════════════════════════════════
# Phase 3: Tokenizer + Generation Config (解决 P3)
# ════════════════════════════════════════════════════════════

tokenizer = AutoTokenizer.from_pretrained(_LORA_DIR)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 3a. 构建 EOS token IDs 列表
_eos_ids = []
if hasattr(model.config, 'eos_token_id'):
    v = model.config.eos_token_id
    _eos_ids = list(v) if isinstance(v, list) else [v]
_im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
if _im_end_id and _im_end_id not in _eos_ids:
    _eos_ids.append(_im_end_id)
EOS_TOKEN_IDS = _eos_ids

# 3b. 重置 generation_config，解决 max_length=32768 冲突
#     之前 model.generation_config.max_length = None 不生效，
#     因为 Qwen2.5 的 generation_config.json 强制设了 max_length=32768。
#     正确做法：用全新的 GenerationConfig 替换，只保留必要参数。
model.generation_config = GenerationConfig(
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=EOS_TOKEN_IDS,
    max_new_tokens=512,
)
print(f'[P3] generation_config reset (no max_length conflict)')
print(f'[P3] EOS token IDs: {EOS_TOKEN_IDS}')

# ════════════════════════════════════════════════════════════
# Phase 4: 断言验证 — 在花 GPU 时间之前确认所有前置条件
# ════════════════════════════════════════════════════════════

_impl = getattr(model.config, '_attn_implementation', 'unknown')
_dtype = next(model.parameters()).dtype
_on_gpu = all(p.device.type == 'cuda' for p in model.parameters())
_cls = type(model).__name__
_alloc = torch.cuda.memory_allocated() / 1e9

print(f'\n{"=" * 55}')
print(f'  INFERENCE MODEL DIAGNOSTICS')
print(f'{"=" * 55}')
print(f'  Model class:       {_cls}')
print(f'  attn_impl:         {_impl}')
print(f'  dtype:             {_dtype}')
print(f'  All on GPU:        {_on_gpu}')
print(f'  GPU mem allocated:  {_alloc:.2f} GB')
print(f'  Is PeftModel:      {hasattr(model, "peft_config")}')
print(f'  EOS IDs:           {EOS_TOKEN_IDS}')
print(f'  gen_config.max_length: {getattr(model.generation_config, "max_length", "NOT SET")}')
print(f'{"=" * 55}')

assert _cls == 'Qwen2ForCausalLM', f'Expected Qwen2ForCausalLM, got {_cls}'
assert _impl == 'sdpa', f'Expected sdpa, got {_impl}'
assert _dtype == torch.float16, f'Expected fp16, got {_dtype}'
assert _on_gpu, 'Some params on CPU!'
assert not hasattr(model, 'peft_config'), 'LoRA not merged!'

# 4b. Forward pass 速度基准
_test_ids = tokenizer('hello', return_tensors='pt').input_ids.to(model.device)
torch.cuda.synchronize()
_t0 = time.time()
with torch.no_grad():
    for _ in range(20):
        model(_test_ids)
torch.cuda.synchronize()
_fwd_ms = (time.time() - _t0) / 20 * 1000
print(f'\n  Forward latency:   {_fwd_ms:.1f} ms/step')
if _fwd_ms > 50:
    print(f'  *** WARNING: {_fwd_ms:.0f}ms too high. Expected <20ms A100, <50ms T4.')
else:
    print(f'  Forward speed OK.')
print()

# ════════════════════════════════════════════════════════════
# Phase 5: Inference helpers
# ════════════════════════════════════════════════════════════

SVG_EXTRACT_RE = re.compile(r'<svg.*?</svg>', flags=re.IGNORECASE | re.DOTALL)

GENERATE_KWARGS = dict(
    max_new_tokens=min(CONFIG['max_new_tokens'], 512),
    do_sample=True,
    temperature=CONFIG['temperature'],
    top_p=CONFIG['top_p'],
    repetition_penalty=max(CONFIG.get('repetition_penalty', 1.05), 1.15),
    eos_token_id=EOS_TOKEN_IDS,
    pad_token_id=tokenizer.pad_token_id,
)
print(f'[P5] max_new_tokens={GENERATE_KWARGS["max_new_tokens"]}, '
      f'repetition_penalty={GENERATE_KWARGS["repetition_penalty"]}')


def extract_svg(text):
    """提取 SVG。如果模型截断（无 </svg>），尝试在最后一个完整元素处强制闭合。"""
    m = SVG_EXTRACT_RE.search(text)
    if m:
        return m.group(0).strip()
    # Fallback: 模型 HIT MAX 生成了 <svg>... 但没来得及输出 </svg>
    idx = text.lower().find('<svg')
    if idx < 0:
        return ''
    partial = text[idx:]
    last_selfclose = partial.rfind('/>')
    last_closetag = partial.rfind('</')
    if last_closetag > last_selfclose:
        end_gt = partial.find('>', last_closetag)
        if end_gt > 0:
            cut = end_gt + 1
        else:
            cut = last_selfclose + 2 if last_selfclose > 0 else -1
    elif last_selfclose > 0:
        cut = last_selfclose + 2
    else:
        # 最后手段：模型卡在 path d="..." 内部，强制关闭 path 和 svg
        # 找到最后一个 <path 或 <... 标签的起始位置，截断 d 属性
        last_open_tag = max(partial.rfind('<path'), partial.rfind('<circle'),
                            partial.rfind('<rect'), partial.rfind('<ellipse'),
                            partial.rfind('<line'), partial.rfind('<polyline'),
                            partial.rfind('<polygon'))
        if last_open_tag > 0:
            recovered = partial[:last_open_tag].rstrip() + '\n</svg>'
            return recovered.strip()
        return ''
    recovered = partial[:cut] + '\n</svg>'
    return recovered.strip()


def _postprocess(decoded, prompt, debug=False):
    """返回 (svg, status)。debug=True 打印每一步的中间结果。"""
    raw = extract_svg(decoded)
    if not raw:
        if debug:
            print(f'    [FAIL] extract_svg returned empty')
            print(f'    [FAIL] decoded text (first 500 chars):')
            print(f'           {decoded[:500]}')
        return fallback_svg(prompt), 'extract_fail'

    cleaned = sanitize_svg(raw)
    if not cleaned:
        if debug:
            print(f'    [FAIL] sanitize_svg returned empty')
            print(f'    [FAIL] raw SVG (first 300 chars): {raw[:300]}')
        return fallback_svg(prompt), 'sanitize_fail'

    ok, reason = validate_svg(cleaned)
    if not ok:
        if debug:
            print(f'    [FAIL] validate_svg: {reason}')
            print(f'    [FAIL] cleaned SVG (first 300 chars): {cleaned[:300]}')
        return fallback_svg(prompt), f'validate_fail:{reason}'

    if debug:
        print(f'    [OK] valid SVG, {len(cleaned)} chars')
    return cleaned, 'ok'


def generate_svg(prompt, debug=False):
    """单条推理。debug=True 打印生成细节。"""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    chat = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(chat, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[-1]

    with torch.no_grad():
        out = model.generate(**inputs, **GENERATE_KWARGS)

    gen_ids = out[0][input_len:]
    n_gen = len(gen_ids)
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    if debug:
        _actual_max = GENERATE_KWARGS['max_new_tokens']
        hit_eos = n_gen < _actual_max
        print(f'  [gen] {n_gen} tokens ({"stopped at EOS" if hit_eos else f"HIT MAX={_actual_max}"})')
        print(f'  [gen] decoded len={len(decoded)}, first 300 chars:')
        print(f'        {decoded[:300]}')

    return _postprocess(decoded, prompt, debug=debug)


def generate_svg_batch(prompts):
    """逐条推理（避免 left-padding 导致的质量劣化）。"""
    all_results = []
    for p in prompts:
        svg, status = generate_svg(p, debug=False)
        all_results.append((svg, status))
    return all_results


print('Inference setup complete. Ready for smoke test.\n')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# Smoke Test with Full Diagnostics
# 用于替换 baseline-qwen2-5.ipynb 的 Smoke Test cell
#
# 相比旧版增加：
# 1. 每条测试 prompt 打印完整生成诊断（token 数、是否 EOS、原始文本）
# 2. 打印 extract/sanitize/validate 每步的失败原因
# 3. 修复旧版 model.base_model.model 在 merge_and_unload 后不存在的 bug
# 4. 用 GENERATE_KWARGS (不含 max_length) 避免警告刷屏
# ═══════════════════════════════════════════════════════════════════

print('=' * 60)
print('SMOKE TEST: Pre-submission Diagnostics')
print('=' * 60)

# ── 1. 环境检查 ──
_impl = getattr(model.config, '_attn_implementation', 'unknown')
_cls = type(model).__name__
_dtype = next(model.parameters()).dtype
_dev = next(model.parameters()).device

print(f'\n[ENV] Model class:       {_cls}')
print(f'[ENV] attn_impl:         {_impl}')
print(f'[ENV] dtype:             {_dtype}')
print(f'[ENV] Device:            {_dev}')
print(f'[ENV] EOS IDs:           {EOS_TOKEN_IDS}')
print(f'[ENV] Batch size:        {CONFIG.get("inference_batch_size", 4)}')
print(f'[ENV] max_new_tokens:    {CONFIG["max_new_tokens"]}')
print(f'[ENV] gen_config.max_length: {getattr(model.generation_config, "max_length", "NOT SET")}')

_problems = []
if _impl != 'sdpa':
    _problems.append(f'attn_implementation={_impl}, expected sdpa')
if _dtype != torch.float16:
    _problems.append(f'dtype={_dtype}, expected fp16')
if hasattr(model, 'peft_config'):
    _problems.append('LoRA not merged (PeftModel still active)')
if getattr(model.generation_config, 'max_length', None) is not None:
    _problems.append(f'generation_config.max_length={model.generation_config.max_length} (will cause warnings)')

if _problems:
    print(f'\n*** {len(_problems)} PROBLEM(S) DETECTED ***')
    for p in _problems:
        print(f'  - {p}')
else:
    print('\n[OK] All environment checks passed.')

# ── 2. 详细单条推理测试 (debug=True) ──
test_prompts = [
    'a simple red circle on white background',
    'a green tree with brown trunk',
    'a blue five-pointed star icon',
]

print(f'\n{"─" * 60}')
print(f'Running {len(test_prompts)} test prompts with FULL DIAGNOSTICS...\n')

single_results = []
t0 = time.time()

for idx, p in enumerate(test_prompts):
    print(f'--- Prompt {idx+1}/{len(test_prompts)}: "{p}" ---')
    t1 = time.time()
    svg, status = generate_svg(p, debug=True)
    elapsed = time.time() - t1
    single_results.append((p, svg, status, elapsed))
    print(f'  Result: [{status}] {elapsed:.1f}s, SVG len={len(svg)}\n')

single_total = time.time() - t0
print(f'Single-mode total: {single_total:.1f}s for {len(test_prompts)} prompts')

# ── 3. EOS 工作状况判断（基于上面 debug=True 的输出已经可见） ──
# 如果所有测试 prompt 的耗时都接近 max_new_tokens / 速度，说明 EOS 从不生成
_avg_time = single_total / max(len(test_prompts), 1)
_expected_max_time = CONFIG['max_new_tokens'] / 30  # 30 tok/s 基准
if _avg_time > _expected_max_time * 0.8:
    print(f'\n*** WARNING: Avg {_avg_time:.1f}s/prompt suggests model always generates max_new_tokens. ***')
    print('    EOS/<|im_end|> may never be generated (training quality issue).')
else:
    print(f'\n[OK] Avg {_avg_time:.1f}s/prompt — model appears to stop before max_new_tokens.')

# ── 4. 批量推理测试 ──
bs = CONFIG.get('inference_batch_size', 4)
print(f'\n{"─" * 60}')
print(f'Running batch test (batch_size={bs})...\n')

t0 = time.time()
batch_results = generate_svg_batch(test_prompts)
batch_elapsed = time.time() - t0

for p, (svg, status) in zip(test_prompts, batch_results):
    print(f'  [{status:>20}] len={len(svg):>5}  prompt="{p}"')

print(f'\n  Batch total: {batch_elapsed:.1f}s for {len(test_prompts)} prompts')

# ── 5. 提交时间估算 ──
per_prompt = batch_elapsed / max(len(test_prompts), 1)
total_prompts = 1000
est_min = per_prompt * total_prompts / 60

print(f'\n{"═" * 60}')
print(f'SUBMISSION ESTIMATE')
print(f'  Speed:      {per_prompt:.2f} s/prompt')
print(f'  Prompts:    {total_prompts}')
print(f'  ETA:        {est_min:.0f} min ({est_min/60:.1f} hours)')
print(f'{"═" * 60}')

# ── 6. 质量汇总 ──
valid = sum(1 for _, s in batch_results if s == 'ok')
print(f'\n[QUALITY] Valid: {valid}/{len(batch_results)}')
for p, (svg, status) in zip(test_prompts, batch_results):
    if status == 'ok':
        print(f'  OK   "{p}" → {len(svg)} chars')
    else:
        print(f'  FAIL "{p}" → {status}')

# ── 7. 最终建议 ──
print(f'\n{"─" * 60}')
if valid == 0 and len(batch_results) > 0:
    print('*** ALL test SVGs failed. Check the diagnostic output above. ***')
    print('    Most common causes:')
    print('    1. Model generates text but not SVGs (training quality issue)')
    print('    2. Model generates SVGs but they fail XML parsing (sanitize_svg)')
    print('    3. Training data format mismatch (check format_chat template)')
    print('    Run the detailed diagnostics above to determine which.')
if est_min > 120:
    print(f'\n*** Estimated {est_min:.0f} min is too slow for Kaggle (limit ~540 min). ***')
elif est_min > 60:
    print(f'\nEstimated {est_min:.0f} min — within limits but could be faster.')
else:
    print(f'\nSpeed looks good for submission.')
print()


---
## 7. Generate Submission

In [ ]:
test_df = pd.read_csv(CONFIG['test_prompts_path'])
print(f'Test prompts: {len(test_df)}')

# Suppress the noisy "max_new_tokens vs max_length" warning printed every generate() call
model.generation_config.max_length = None

all_prompts = test_df['prompt'].tolist()
all_ids = test_df['id'].tolist()
BS = CONFIG.get('inference_batch_size', 8)

rows = []
stats = Counter()
t0 = time.time()

for i in range(0, len(all_prompts), BS):
    batch_prompts = all_prompts[i:i + BS]
    batch_ids = all_ids[i:i + BS]
    batch_results = generate_svg_batch(batch_prompts)

    for pid, (svg, status) in zip(batch_ids, batch_results):
        stats[f'status:{status}'] += 1
        if status != 'ok':
            stats['fallback'] += 1
        else:
            stats['valid'] += 1
        stats['total_len'] += len(svg)
        rows.append({'id': pid, 'svg': svg})

    done = min(i + BS, len(all_prompts))
    if done % 50 < BS or done >= len(all_prompts):
        elapsed = time.time() - t0
        rate = done / max(elapsed, 1)
        eta = (len(all_prompts) - done) / max(rate, 0.01)
        print(f'  [{done}/{len(all_prompts)}] {elapsed:.0f}s '
              f'({rate:.2f} it/s, ETA {eta/60:.0f}min)  '
              f'valid={stats["valid"]}  fallback={stats["fallback"]}')

sub_df = pd.DataFrame(rows)
sub_df.to_csv(CONFIG['submission_path'], index=False)

elapsed_min = (time.time() - t0) / 60
n = len(sub_df)
print(f'\nSubmission saved: {CONFIG["submission_path"]}')
print(f'Total rows:    {n}')
print(f'Valid:         {stats["valid"]} ({100*stats["valid"]/max(n,1):.1f}%)')
print(f'Fallback:      {stats["fallback"]} ({100*stats["fallback"]/max(n,1):.1f}%)')
print(f'Avg SVG len:   {stats["total_len"]/max(n,1):.0f} chars')
print(f'Runtime:       {elapsed_min:.1f} min')

print(f'\n--- Failure distribution ---')
for k, v in sorted(stats.items()):
    if k.startswith('status:'):
        print(f'  {k}: {v}')

sub_df.head()

---
## Summary & Notes

**What this baseline does:**

- Loads competition `train.csv` (no external datasets), normalizes SVGs to 256×256, filters by quality (≤2 500 chars)
- Fine-tunes Qwen2.5-3B-Instruct with QLoRA (`r=16`, 4-bit, `packing=False`, `response_template` for loss masking)
- Batched inference (`inference_batch_size=4`) with conservative decoding (`temperature=0.3`, `top_p=0.8`)
- Full post-processing chain: extract → sanitize → validate → fallback
- Tracks per-layer failure reasons (`extract_fail`, `sanitize_fail`, `validate_fail:<reason>`)

**Two-phase design:**

- **Phase A (Training):** Cells 1-15. Requires internet to load model weights. Saves LoRA adapter to `output_dir`.
- **Phase B (Inference):** Cells 16-21. Reloads model without Unsloth patches for clean generation.

**Key parameters to tune next:**

| Parameter | Current | Try |
|---|---|---|
| `max_train_samples` | 12 000 | 20 000-29 000 |
| `max_svg_train_chars` | 2 500 | 3 000-4 000 (with monitoring) |
| `lora_r` | 16 | 32, 64 |
| `num_train_epochs` | 1 | 2-3 |
| `temperature` | 0.3 | 0.5-0.7 (after valid rate ≥ 80%) |
| `inference_batch_size` | 4 | 8 (if VRAM allows) |